# Ajustar um modelo com Microsoft Foundry

Este notebook segue o fluxo de trabalho atual de [Ajuste fino do Microsoft Foundry](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning?WT.mc_id=academic-105485-koreyst). Ele usa o **SDK Python da OpenAI (v1)** apontando para o endpoint `/openai/v1/` do seu recurso Foundry, então o mesmo código também funciona na plataforma OpenAI com apenas a configuração do cliente alterada.

> **Pré-requisitos**
> - Um projeto [Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst), com a função **Foundry Owner** (necessário para ajustar e implantar).
> - `pip install "openai>=1.0" python-dotenv`
> - Um arquivo `.env` com `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY` (veja o [guia de configuração do curso](./../../../00-course-setup/02-setup-local.md?WT.mc_id=academic-105485-koreyst)).
> - Um modelo base atualmente suportado, como `gpt-4.1-mini` (veja a [lista de modelos ajustáveis](https://learn.microsoft.com/azure/ai-foundry/foundry-models/concepts/models-sold-directly-by-azure?WT.mc_id=academic-105485-koreyst#fine-tuning-models)).

O ajuste fino melhora um modelo base ao re-treiná-lo com exemplos adicionais relevantes para sua tarefa. Técnicas de engenharia de prompt como _few-shot learning_ e _retrieval-augmented generation_ aprimoram o prompt com dados relevantes, mas são limitadas pela janela de contexto do modelo.

Com o ajuste fino, você re-treina o próprio modelo (usando muitos mais exemplos do que cabem na janela de contexto) e implanta uma versão _customizada_ que já não precisa desses exemplos na hora da inferência. Isso melhora a qualidade, libera a janela de contexto, e pode reduzir custo e latência ao encurtar os prompts. Por baixo dos panos, o Foundry usa **LoRA (adaptação de baixa ordem)** para treinamento eficiente.

O Foundry suporta três técnicas: **ajuste fino supervisionado (SFT)** - usado neste tutorial - além de **DPO** (alinhamento de preferência) e **RFT** (ajuste fino por reforço). O fluxo de trabalho tem quatro etapas:

1. Prepare e envie seus dados de treinamento **e validação**.
2. Execute o trabalho de treinamento para criar um modelo ajustado.
3. Monitore o trabalho, revise as métricas e escolha um checkpoint.
4. Implemente o modelo ajustado e use-o para inferência.

Neste tutorial, ajustamos o `gpt-4.1-mini` com SFT para construir um chatbot que responde perguntas sobre a tabela periódica com uma quadra.

---


### Passo 1.1: Prepare Seu Conjunto de Dados

Vamos construir um chatbot que te ajude a entender a tabela periódica dos elementos respondendo perguntas sobre um elemento com um limerique. Neste _simples_ tutorial, vamos apenas criar um conjunto de dados para treinar o modelo com alguns exemplos de respostas que mostram o formato esperado dos dados. Em um caso de uso real, você precisaria criar um conjunto de dados com muito mais exemplos. Você também pode usar um conjunto de dados aberto (para seu domínio de aplicação) se existir, e reformatá-lo para uso no fine-tuning.

Como estamos focando no `gpt-4.1-mini` e buscando uma resposta de turno único (chat completion), podemos criar exemplos usando [este formato sugerido](https://platform.openai.com/docs/guides/fine-tuning/preparing-your-dataset?WT.mc_id=academic-105485-koreyst) refletindo os requisitos de chat completion da OpenAI. Se você espera conteúdo de conversação multi-turno, você usaria o [formato de exemplo multi-turno](https://platform.openai.com/docs/guides/fine-tuning/multi-turn-chat-examples?WT.mc_id=academic-105485-koreyst) que inclui um parâmetro `weight` para sinalizar quais mensagens devem ser usadas (ou não) no processo de fine-tuning.

Vamos usar o formato mais simples de turno único para o nosso tutorial aqui. Os dados estão no [formato jsonl](https://jsonlines.org/?WT.mc_id=academic-105485-koreyst) com 1 registro por linha, cada um representado como um objeto formatado em JSON. O trecho abaixo mostra 2 registros como amostra — veja [training-data.jsonl](../../../../../18-fine-tuning/python/openai/training-data.jsonl) para o conjunto completo de amostras (10 exemplos) que usaremos no nosso tutorial de fine-tuning. **Nota:** Cada registro _deve_ ser definido em uma única linha (não dividido em múltiplas linhas como é típico em um arquivo JSON formatado)

```json
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Gallium"}, {"role": "assistant", "content": "Gallium, oh gallium, so light - Melts in your hand, oh what a sight - At 86 degrees - Its liquid with ease - And in semiconductors, it's out of sight"}]}
{ "messages": [{"role": "system", "content": "Elle is a factual chatbot that answers questions about elements in the periodic table with a limerick"}, {"role": "user", "content": "Tell me about Hydrogen"}, {"role": "assistant", "content": "Hydrogen, the first in the line - The lightest of all, so divine - It's in water, you see - And in stars, it's the key - The universe's most common sign"}]}
```

Em um caso de uso real você precisará de um conjunto muito maior de exemplos para bons resultados — o equilíbrio será entre qualidade das respostas e o tempo/custo do fine-tuning. Estamos usando um conjunto pequeno para podermos concluir o fine-tuning rapidamente e ilustrar o processo. Veja [este exemplo do OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) para um tutorial de fine-tuning mais complexo.


---

### Etapa 1.2: Faça o upload do seu conjunto de dados

Faça o upload dos seus arquivos de treinamento e validação para o Foundry usando a Files API (`purpose="fine-tune"`). Fornecer um **arquivo de validação** permite que o Foundry reporte a perda de validação para que você possa detectar overfitting.

Antes de executar o código abaixo, certifique-se de que você:
 - Instalou o SDK: `pip install "openai>=1.0" python-dotenv`
 - Criou um arquivo `.env` com `AZURE_OPENAI_ENDPOINT` e `AZURE_OPENAI_API_KEY`

O código cria um cliente OpenAI apontando para o endpoint `/openai/v1/` do seu recurso Foundry, depois faz o upload de ambos os arquivos e imprime seus IDs.


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

# Point the OpenAI SDK at your Microsoft Foundry resource's /openai/v1/ endpoint.
# (For the OpenAI platform instead, use: client = OpenAI()  with OPENAI_API_KEY set.)
client = OpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
)

training_response = client.files.create(
    file=open("./training-data.jsonl", "rb"), purpose="fine-tune"
)
validation_response = client.files.create(
    file=open("./validation-data.jsonl", "rb"), purpose="fine-tune"
)

training_file_id = training_response.id
validation_file_id = validation_response.id

print("Training file ID:", training_file_id)
print("Validation file ID:", validation_file_id)


---

### Passo 2.1: Criar o trabalho de Fine-tuning com o SDK


In [ ]:
# Create the fine-tuning job.
# trainingType "GlobalStandard" is the recommended tier (lower cost, faster queue).
# Use "Standard" if you need regional data residency, or "Developer" for cheap experiments.
job = client.fine_tuning.jobs.create(
    model="gpt-4.1-mini",              # base model to fine-tune
    training_file=training_file_id,
    validation_file=validation_file_id,
    suffix="elements-limerick",        # helps you identify the resulting model
    seed=105,                          # makes the run reproducible
    extra_body={"trainingType": "GlobalStandard"},
)

job_id = job.id
print("Fine-tuning Job ID:", job_id)
print("Status:", job.status)


---

### Etapa 2.2: Verifique o status do trabalho

Aqui estão algumas coisas que você pode fazer com a API `client.fine_tuning.jobs`:
- `client.fine_tuning.jobs.list(limit=<n>)` - listar os últimos n trabalhos de fine-tuning
- `client.fine_tuning.jobs.retrieve(<job_id>)` - obter detalhes de um trabalho específico
- `client.fine_tuning.jobs.cancel(<job_id>)` - cancelar um trabalho
- `client.fine_tuning.jobs.list_events(fine_tuning_job_id=<job_id>, limit=<n>)` - listar eventos do trabalho
- `client.fine_tuning.jobs.checkpoints.list(<job_id>)` - listar checkpoints implantáveis (as últimas poucas épocas)

Quando o trabalho começa, o Foundry primeiro _valida o arquivo de treinamento_ para garantir que os dados estejam no formato correto. O treinamento então é executado e pode levar de minutos a horas, dependendo do modelo e do tamanho do conjunto de dados.


In [ ]:
# List the last 10 fine-tuning jobs
client.fine_tuning.jobs.list(limit=10)

# Retrieve the current state of your fine-tuning job
client.fine_tuning.jobs.retrieve(job_id)

# List up to 10 events from the job
client.fine_tuning.jobs.list_events(fine_tuning_job_id=job_id, limit=10)


In [ ]:
# Track the job status until it is complete
response = client.fine_tuning.jobs.retrieve(job_id)

print("Job ID:", response.id)
print("Status:", response.status)
print("Trained Tokens:", response.trained_tokens)


---

### Etapa 2.3: Acompanhar eventos para monitorar o progresso


In [ ]:
# Track progress in a more granular way by checking events.
# Re-run this cell until you see "The job has successfully completed".
response = client.fine_tuning.jobs.list_events(job_id)

events = response.data
events.reverse()

for event in events:
    print(event.message)


In [ ]:
# When the job finishes, the last few epochs are available as deployable checkpoints.
# Best practice: don't blindly deploy the final epoch - review the checkpoints and pick
# the one that generalizes best (watch train_loss vs. valid_loss and token accuracy).
checkpoints = client.fine_tuning.jobs.checkpoints.list(job_id)
for cp in checkpoints.data:
    print(cp.fine_tuned_model_checkpoint, "| step:", cp.step_number)


### Etapa 2.4: Revisar status, métricas e pontos de verificação no portal Foundry


Você também pode acompanhar o trabalho no [portal Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) em **Build > Fine-tuning**. Selecione seu trabalho para ver seu status, eventos de treinamento, hiperparâmetros e métricas. Observe estas métricas:

- `train_loss` e `full_valid_loss` - devem diminuir ao longo do tempo.
- `train_mean_token_accuracy` e `full_valid_mean_token_accuracy` - devem aumentar.

Se as curvas de treinamento e validação divergirem, você pode estar com overfitting - tente menos épocas ou um multiplicador de taxa de aprendizado menor. A ilustração abaixo mostra o tipo de painel de status, mensagem e métricas que você verá (a interface exata varia conforme o provedor).

![Status do trabalho de fine-tuning](../../../../../translated_images/pt-BR/fine-tuned-model-status.563271727bf7bfba.webp)


Você também pode visualizar as mensagens de status e métricas rolando para baixo no painel visual como mostrado:

| Messages | Metrics |
|:---|:---|
| ![Messages](../../../../../translated_images/pt-BR/fine-tuned-messages-panel.4ed0c2da5ea1313b.webp) |  ![Metrics](../../../../../translated_images/pt-BR/fine-tuned-metrics-panel.700d7e4995a65229.webp)|


---

### Passo 3.1: Obtenha o id do modelo ajustado

Quando o trabalho for concluído com sucesso, `response.fine_tuned_model` contém o id do seu modelo personalizado (por exemplo, `gpt-4.1-mini-2025-04-14.ft-...`). No Azure, você deve **implantar** esse modelo antes de poder usá-lo.


In [ ]:
# Retrieve the id of the fine-tuned model once the job has succeeded
response = client.fine_tuning.jobs.retrieve(job_id)
fine_tuned_model_id = response.fine_tuned_model
print("Fine-tuned Model ID:", fine_tuned_model_id)


### Etapa 3.2: Implantar o modelo ajustado

Diferente da plataforma OpenAI (onde você pode chamar diretamente o id do modelo ajustado), o Microsoft Foundry exige que você **implante** o modelo primeiro. A maneira mais fácil é pelo [portal Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst): vá em **Build > Fine-tuning**, selecione seu trabalho finalizado e escolha **Deploy**. Dê um nome para a implantação (por exemplo, `elements-limerick`) - este nome de implantação é o que você usará no código.

Você também pode implantar programaticamente com a API REST/ARM do plano de controle; veja o [guia de implantação](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/fine-tuning-deploy?WT.mc_id=academic-105485-koreyst). Lembre-se que um modelo personalizado implantado é cobrado por hora, e uma implantação inativa é removida após 15 dias.


In [ ]:
# Once deployed, call your fine-tuned model by its DEPLOYMENT name via the Responses API.
# (On the OpenAI platform you'd pass fine_tuned_model_id directly instead.)
deployment_name = "elements-limerick"  # the name you gave the deployment in Foundry

completion = client.responses.create(
    model=deployment_name,
    input=[
        {"role": "system", "content": "You are Elle, a factual chatbot that answers questions about elements in the periodic table with a limerick"},
        {"role": "user", "content": "Tell me about Strontium"},
    ],
    store=False,
)
print(completion.output_text)


---

### Passo 3.3: Teste seu modelo ajustado no playground do Foundry

Você também pode testar o modelo implantado no [portal Microsoft Foundry](https://ai.azure.com?WT.mc_id=academic-105485-koreyst) **Playground** - escolha sua implantação ajustada no menu suspenso de modelos e experimente um prompt. Use a **mesma mensagem do sistema** com a qual você treinou; uma mensagem do sistema diferente pode alterar o comportamento.

> **Dica:** Compare o modelo ajustado com o `gpt-4.1-mini` base lado a lado. Observe como o modelo ajustado retorna respostas no formato de limerique dos seus exemplos, enquanto o modelo base apenas segue o prompt do sistema.

**Este é um exemplo deliberadamente simples para mostrar o processo, não um conjunto de dados do mundo real.** Na produção, você ajustaria com dados reais (por exemplo, um catálogo de produtos para atendimento ao cliente), onde a diferença de qualidade é muito mais aparente - e atingir essa qualidade apenas com engenharia de prompt custaria muitos mais tokens por chamada. Para um exemplo mais completo, veja o [guia de ajuste fino do OpenAI Cookbook](https://github.com/openai/openai-cookbook/blob/main/examples/How_to_finetune_chat_models.ipynb?WT.mc_id=academic-105485-koreyst) e o [tutorial de ajuste fino do Foundry](https://learn.microsoft.com/azure/ai-foundry/openai/tutorials/fine-tune?WT.mc_id=academic-105485-koreyst).

---


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido usando o serviço de tradução por IA [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, por favor, esteja ciente de que traduções automatizadas podem conter erros ou imprecisões. O documento original em seu idioma nativo deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas decorrentes do uso desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
